Building the recommender!!

Our goal is to make a first draft working recommender. We can give it a book and it will return other books from the dataset that are most similar to it.
We'll use content-based filtering: recommendations are based on book characteristics, rather than on what other readers liked.

In [1]:
#load in our packages

import pandas as pd
from pathlib import Path
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
#load in our dataset

df = pd.read_csv("../data/processed/books_clean.csv")

df.head()

,book_id,title,author,genres,publication_year,page_count,average_rating,rating_count,description
0,B001,Dune,Frank Herbert,Science Fiction;Politics;Adventure,1965,412,4.27,850000,A young nobleman becomes entangled in a strugg...
1,B002,Hyperion,Dan Simmons,Science Fiction;Space Opera;Adventure,1989,482,4.22,185000,Seven travelers journey toward a mysterious wo...
2,B003,The Left Hand of Darkness,Ursula K. Le Guin,Science Fiction;Politics;Literary,1969,304,4.08,190000,An envoy visits a distant planet and must navi...
3,B004,The Dispossessed,Ursula K. Le Guin,Science Fiction;Politics;Literary,1974,387,4.23,170000,A physicist moves between contrasting worlds w...
4,B005,Foundation,Isaac Asimov,Science Fiction;Space Opera;Politics,1951,296,4.17,500000,A mathematician predicts the collapse of a vas...


In [3]:
#load in the features

combined_features = load_npz(
    "../data/processed/book_combined_features.npz"
)

print("Books:", len(df))
print("Feature matrix shape:", combined_features.shape)

Books: 99
Feature matrix shape: (99, 776)


We'll use cosine similarity, which measures how similar two feature vectors are, to compare every book with every other book.

- A score closer to 1 means the books are more similar in their feature representation.
- A score closer to 0 means they have less overlap in that representation.
- Each book will have a similarity score with itself, typically 1.

This is a measure of similarity, not a guarantee that a reader will enjoy a book.

In [4]:
similarity_matrix = cosine_similarity(combined_features)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (99, 99)


Let's inspect the similarity scores. The first result will usually be the book itself, which is to be expected. In a later step, we'll exclude the input book from the recommendations.

In [5]:
book_index = 0

scores = pd.Series(
    similarity_matrix[book_index],
    index=df.index
)

scores.sort_values(ascending=False).head(10)

0     1.000000
2     0.230769
4     0.222375
1     0.167248
3     0.167248
5     0.153846
8     0.153846
31    0.153846
20    0.153846
72    0.153846
dtype: float64

Now we can build a first recommendation function.

We'll start with a function that accepts a book title and returns the most similar books. The function should:
1. Find the selected book.
2. Retrieve its similarity scores.
3. Exclude the selected title and duplicate entries with the same title.
4. Rank the remaining books from highest to lowest similarity.
5. Return a readable table.

In [6]:
def recommend_books(book_title, top_n=5): #we'll return 5 recommendations

    #Find every row in the dataset whose title matches the title the user entered, ignoring capitalization.
    matches = df.index[
        df["title"].str.casefold() == book_title.casefold()
    ].tolist()

    #in case a user enters a book not in the dataset
    if not matches:
        raise ValueError(f"Book not found: {book_title}")

    book_index = matches[0] #if there are duplicate titles, this function only acknowledges the first appearance of the title

    #get the similarity scores for the book entered
    scores = pd.Series(
        similarity_matrix[book_index],
        index=df.index
    )

    # Exclude the input title, including duplicate rows
    scores = scores[
        df["title"].str.casefold() != book_title.casefold()
    ]

    #ranks the remaining books and returns the most similar book
    top_indices = scores.sort_values(
        ascending=False
    ).head(top_n).index

    #retrieve book info
    results = df.loc[
        top_indices,
        ["book_id", "title", "author", "genres", "average_rating"]
    ].copy() #creating a separate copy of data

    #appends similarity score to results table
    results["similarity_score"] = scores.loc[top_indices].values

    return results.reset_index(drop=True)


Let's take multiple books as input instead of just one.

In [7]:
def recommend_books_for_multiple(books_read, top_n=5): #we'll return 5 recommendations

    #make sure books are in dataset
    titles =[]
    indices = []

    for title in books_read:
        n_title = title.casefold()

        #skips the rest of the loop if the title is already stored
        if n_title in titles:
            continue

        matches = df.index[
            df["title"].str.casefold() == n_title
        ].tolist()

        #in case a user enters a book not in the dataset
        if not matches:
            raise ValueError(f"Book not found: {title}")

        indices.append(matches[0])
        titles.append(n_title)

    if not indices:
        raise ValueError("Please provide at least one book title.")

    average_scores = similarity_matrix[indices].mean(axis=0)
    scores = pd.Series(average_scores, index=df.index)

    # Exclude books the user has already read
    scores = scores[
        ~df["title"].str.casefold().isin(titles) #~ means not
    ]

    #ranks the remaining books and returns the most similar book
    top_indices = scores.sort_values(
        ascending=False
    ).head(top_n).index

    #retrieve book info
    results = df.loc[
        top_indices,
        ["book_id", "title", "author", "genres", "average_rating"]
    ].copy() #creating a separate copy of data

    #appends similarity score to results table
    results["similarity_score"] = scores.loc[top_indices].values

    return results.reset_index(drop=True)


In [8]:

books_read = [
    "The Wise Man's Fear",
    "The Hobbit",
]

recommendations = recommend_books_for_multiple(
    books_read,
    top_n=5
)

recommendations


,book_id,title,author,genres,average_rating,similarity_score
0,B020,The Fellowship of the Ring,J. R. R. Tolkien,Fantasy;Epic;Adventure,4.38,0.188955
1,B074,Into the Wild,Jon Krakauer,Biography;Adventure;Nature,4.00,0.184056
2,B014,The Name of the Wind,Patrick Rothfuss,Fantasy;Magic;Coming of Age,4.50,0.177360
3,B028,A Wizard of Earthsea,Ursula K. Le Guin,Fantasy;Magic;Coming of Age,4.28,0.141646
4,B022,The Lies of Locke Lamora,Scott Lynch,Fantasy;Crime;Adventure,4.28,0.135673
